# 面试问题：LLM 训练中的 DP、TP、PP、Sequence Parallel 分别解决什么，怎样规划 3D 并行？

**一句话回答**：DP 复制模型并切 batch，TP 在层内切矩阵/attention head，PP 沿层切 stage，Sequence Parallel 再切 token 维激活。选择不是把 GPU 数随意因式分解，而是同时满足张量整除、单卡显存、节点拓扑、通信带宽、micro-batch 和 pipeline bubble。

本 Notebook 用 NumPy 验证 column/row tensor parallel 的数值等价，手写坐标映射、流水线时序、内存/通信估算和拓扑感知枚举器。


In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
import itertools,math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

SEED128=12801; rng128=np.random.default_rng(SEED128)  # 计算并保存当前步骤的中间状态。
assert SEED128==12801  # 用受控断言验证关键不变量。
assert 2*4*2==16  # 用受控断言验证关键不变量。
assert np.isfinite(rng128.normal())  # 用受控断言验证关键不变量。


## 1. 并行坐标是显式契约

设世界大小为 `DP×TP×PP`。同一 DP 组处理不同样本后同步梯度；同一 TP 组共同完成一层；PP rank 持有连续层。rank 到坐标的映射必须稳定写入 checkpoint，否则扩缩容和故障恢复会把 shard 拼错。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Mesh128:  # 定义承载本节状态与行为的数据结构。
    dp:int; tp:int; pp:int  # 执行当前语句以推进本节示例。
    def rank(self,d,t,p): return (d*self.pp+p)*self.tp+t  # 定义本节可复用的核心函数。
    def coord(self,r): return (r//(self.pp*self.tp),(r//self.tp)%self.pp,r%self.tp)  # 定义本节可复用的核心函数。
mesh128=Mesh128(2,4,2)  # 计算并保存当前步骤的中间状态。
assert mesh128.rank(1,3,1)==15  # 用受控断言验证关键不变量。
assert mesh128.coord(15)==(1,1,3)  # 用受控断言验证关键不变量。
assert len({mesh128.rank(*c) for c in itertools.product(range(2),range(4),range(2))})==16  # 用受控断言验证关键不变量。


## 2. Tensor Parallel 的本质是切线性层并插入 collective

Column Parallel 按输出列切 `W`，各 rank 结果 concat；Row Parallel 按输入行切 `X/W`，各 rank 局部乘积再 sum。它们组合后可避免在 MLP 中间立刻 all-gather。下面只用数组切片证明结果与 dense GEMM 一致。


In [ ]:
X128=rng128.normal(size=(3,8)); W128=rng128.normal(size=(8,12)); dense128=X128@W128  # 计算并保存当前步骤的中间状态。
def column_parallel128(X,W,tp): return np.concatenate([X@s for s in np.array_split(W,tp,axis=1)],axis=1)  # 定义本节可复用的核心函数。
def row_parallel128(X,W,tp): return sum(xs@ws for xs,ws in zip(np.array_split(X,tp,axis=1),np.array_split(W,tp,axis=0)))  # 定义本节可复用的核心函数。
assert np.allclose(column_parallel128(X128,W128,4),dense128)  # 用受控断言验证关键不变量。
assert np.allclose(row_parallel128(X128,W128,4),dense128)  # 用受控断言验证关键不变量。
assert column_parallel128(X128,W128,4).shape==(3,12)  # 用受控断言验证关键不变量。


## 3. Attention 与隐藏维度必须可整除

TP 常按 attention head 和 MLP 中间维度切。若 `num_heads % TP != 0`，需要不均匀分片或改变并行度；GQA 还要求 KV head 的复制/切分策略明确。Sequence Parallel 可让 LayerNorm、dropout 等 token 维激活分摊，但会引入 reduce-scatter/all-gather。


In [ ]:
def valid_tp128(hidden,heads,kv_heads,tp):  # 定义本节可复用的核心函数。
    return hidden%tp==0 and heads%tp==0 and (kv_heads%tp==0 or tp%kv_heads==0)  # 返回当前分支计算出的结果。
assert valid_tp128(4096,32,8,4)  # 用受控断言验证关键不变量。
assert not valid_tp128(4096,30,8,4)  # 用受控断言验证关键不变量。
local128={"hidden":4096//4,"q_heads":32//4,"kv_mode":"shard"}  # 计算并保存当前步骤的中间状态。
assert local128=={"hidden":1024,"q_heads":8,"kv_mode":"shard"}  # 用受控断言验证关键不变量。


## 4. Pipeline Parallel 用 micro-batch 换 bubble

最简单 GPipe 调度先完成所有 forward 再 backward，stage 需要等待填充/排空。近似 bubble fraction 为 `(PP-1)/(micro_batches+PP-1)`；micro-batch 越多 bubble 越小，但激活、调度开销和有效 batch 约束会变化。1F1B 可降低峰值激活，却不消除跨 stage 通信。


In [ ]:
def bubble128(pp,micro): return (pp-1)/(micro+pp-1)  # 定义本节可复用的核心函数。
def gpipe_forward_slots128(pp,micro): return [[m+s for m in range(micro)] for s in range(pp)]  # 定义本节可复用的核心函数。
slots128=gpipe_forward_slots128(3,4)  # 计算并保存当前步骤的中间状态。
assert slots128[2][0]==2 and slots128[0][-1]==3  # 用受控断言验证关键不变量。
assert bubble128(4,16)<bubble128(4,4)  # 用受控断言验证关键不变量。
assert math.isclose(bubble128(1,8),0.0)  # 用受控断言验证关键不变量。


## 5. Data Parallel 同步的是同一参数语义的梯度

每个 DP rank 在不同 micro-batch 上计算梯度，collective 求和后除以全局样本数。若各 rank 有效 token 数不同，直接平均 rank gradient 会偏置；应按 token/sample count 加权。梯度累积还要求所有 rank 在同一个 optimizer step 边界同步。


In [ ]:
grads128=[np.array([2.,4.])/2,np.array([9.,3.])/3]; counts128=[2,3]  # 计算并保存当前步骤的中间状态。
weighted128=sum(g*n for g,n in zip(grads128,counts128))/sum(counts128)  # 计算并保存当前步骤的中间状态。
naive128=sum(grads128)/len(grads128)  # 计算并保存当前步骤的中间状态。
assert np.allclose(weighted128,[2.2,1.4])  # 用受控断言验证关键不变量。
assert not np.allclose(weighted128,naive128)  # 用受控断言验证关键不变量。
assert sum(counts128)==5  # 用受控断言验证关键不变量。


## 6. 规划器必须同时估参数、优化器、梯度和激活

TP/PP 分摊模型张量，DP 本身会复制；ZeRO 会改变这一点。激活与 `micro_batch×sequence×hidden×layers_per_stage` 相关，并受重计算影响。估算时为临时 buffer、通信 bucket、碎片和 kernel workspace 留余量，不能刚好卡满 HBM。


In [ ]:
def memory_gb128(params,tp,pp,micro,seq,hidden,layers,recompute=.5):  # 定义本节可复用的核心函数。
    model=params*(2+2+8)/(tp*pp) # bf16 weight/grad + fp32 Adam m,v；中文说明：该行遵循既定约束。
    activ=micro*seq*hidden*(layers/pp)*2*recompute*10  # 计算并保存当前步骤的中间状态。
    return (model+activ)/1e9  # 返回当前分支计算出的结果。
mem_a128=memory_gb128(7e9,4,2,1,2048,4096,32)  # 计算并保存当前步骤的中间状态。
assert mem_a128>0  # 用受控断言验证关键不变量。
assert memory_gb128(7e9,8,2,1,2048,4096,32)<mem_a128  # 用受控断言验证关键不变量。
assert memory_gb128(7e9,4,2,2,2048,4096,32)>mem_a128  # 用受控断言验证关键不变量。


## 7. 高频 collective 尽量留在高速域

TP 每层都有通信，通常优先放在 NVLink/NVSwitch 节点内；PP 通信频率更低，可跨节点；DP 的大梯度同步可与反向计算 overlap。下面的简化代价函数惩罚跨节点 TP，并同时考虑 bubble 和显存，展示枚举而非拍脑袋选 `TP=8`。


In [ ]:
def plans128(world,gpus_per_node,hidden,heads,kv_heads,micro):  # 定义本节可复用的核心函数。
    out=[]  # 计算并保存当前步骤的中间状态。
    for dp in range(1,world+1):  # 遍历输入元素以累积或检查结果。
        for tp in range(1,world+1):  # 遍历输入元素以累积或检查结果。
            if world%(dp*tp): continue  # 按当前条件选择后续控制路径。
            pp=world//(dp*tp)  # 计算并保存当前步骤的中间状态。
            if valid_tp128(hidden,heads,kv_heads,tp):  # 按当前条件选择后续控制路径。
                cross=max(0,tp-gpus_per_node); out.append((cross*10+bubble128(pp,micro)+.02*pp,Mesh128(dp,tp,pp)))  # 计算并保存当前步骤的中间状态。
    return sorted(out,key=lambda x:x[0])  # 返回当前分支计算出的结果。
ranked128=plans128(16,8,4096,32,8,16)  # 计算并保存当前步骤的中间状态。
assert ranked128  # 用受控断言验证关键不变量。
assert ranked128[0][0]<=ranked128[-1][0]  # 用受控断言验证关键不变量。
assert all(m.dp*m.tp*m.pp==16 for _,m in ranked128)  # 用受控断言验证关键不变量。


## 8. 发布前验证数值、性能与恢复合同

至少做单卡与分布式 loss/gradient 对齐、不同 micro-batch 等价、吞吐/MFU、显存峰值、慢 rank、collective timeout 和故障恢复。checkpoint manifest 要记录 mesh、层到 stage、参数切分轴、padding 和 optimizer shard；改变世界大小必须显式 reshard。


In [ ]:
manifest128={"world":16,"mesh":{"dp":2,"tp":4,"pp":2},"layer_stage":[0]*16+[1]*16,"format":1}  # 计算并保存当前步骤的中间状态。
digest128=__import__("hashlib").sha256(str(manifest128).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert len(manifest128["layer_stage"])==32  # 用受控断言验证关键不变量。
assert manifest128["mesh"]["dp"]*manifest128["mesh"]["tp"]*manifest128["mesh"]["pp"]==manifest128["world"]  # 用受控断言验证关键不变量。
assert len(digest128)==64  # 用受控断言验证关键不变量。


## 面试总结

一条完整主线是：**先定全局 batch/序列和 HBM → 枚举 `DP×TP×PP` → 检查 hidden/head/KV-head 整除 → TP 放高速互联 → 用 micro-batch 控 PP bubble → 计算激活与模型状态峰值 → 按有效 token 加权 DP 梯度 → 实测 MFU/通信/长尾 → checkpoint 固化 mesh 并验证 reshard**。并行度是性能、内存和故障域的联合设计。

延伸阅读：[Megatron 3D Parallelism](https://arxiv.org/abs/2104.04473)、[Megatron-LM](https://arxiv.org/abs/1909.08053)、[GPipe](https://arxiv.org/abs/1811.06965)。
